# Clinical Validation of 2D Mammography AI — CBIS-DDSM

A reproducible **standalone (black-box) validation** of a pretrained breast-cancer classifier on the CBIS-DDSM test split.

The pipeline never trains or modifies the model — it *measures* it against a biopsy-proven reference standard, which is what clinical validation (as opposed to model development) is.

**Dataset:** CBIS-DDSM (Curated Breast Imaging Subset of DDSM), Kaggle JPEG mirror — biopsy-proven pathology, BI-RADS assessment, breast density, lesion type.
**Model:** an ImageNet-pretrained CNN fine-tuned on the CBIS-DDSM training split (swap in any model by writing one `ImageClassifier`).
**Output:** an HTML validation report — ROC/AUC with DeLong CIs, operating points, calibration, screening behaviour, breast-density subgroup analysis, and an AI-vs-radiologist (BI-RADS) comparison.

## 1 · Environment setup
Runs on **Google Colab** or **Kaggle**.

**On Kaggle:** in the *Notebook options* panel set *Accelerator* to a **GPU** and turn *Internet* **On** — both are required. Internet lets the notebook install packages, fetch the model, and clone the project code.

The next cell installs dependencies; the one after it auto-clones the `mammoval` package if it is not already present.

In [ ]:
!pip install -q transformers timm pillow
# torch / torchvision / pydicom are pre-installed on Colab and Kaggle.

In [ ]:
# Make the `mammoval` package importable. Auto-clones the project repo if it
# is not already present (Internet must be ON for the clone on Kaggle).
import os, sys, glob, subprocess

REPO_URL = 'https://github.com/Joana-Mansa/breast-ai-clinical-validation.git'

def _find_mammoval():
    cands = ['.', 'breast-ai-clinical-validation',
             '/content/breast-ai-clinical-validation']
    cands += glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
    cands += glob.glob('/kaggle/working/*')
    for c in cands:
        if c and os.path.isdir(os.path.join(c, 'mammoval')):
            return os.path.abspath(c)
    return None

loc = _find_mammoval()
if loc is None:
    print('cloning', REPO_URL, '...')
    subprocess.run(['git', 'clone', '-q', REPO_URL], check=False)
    loc = _find_mammoval()
if loc is None:
    raise RuntimeError('mammoval not found and clone failed - turn Internet ON, '
                       'or add the project folder as a Kaggle dataset (+ Add Input).')
sys.path.insert(0, loc)
import mammoval
print('mammoval', mammoval.__version__, 'ready (from ' + loc + ')')

## 2 · Get the CBIS-DDSM dataset
**On Kaggle (easiest):** click **+ Add Input**, search *"CBIS-DDSM Breast Cancer Image Dataset"* (by *awsaf49*) and add it — it mounts read-only, no download or credentials needed.

**On Colab:** the cell downloads the ~6 GB JPEG mirror; it will prompt you to upload `kaggle.json` (Kaggle → Settings → Create New API Token).

In [ ]:
import os, glob
ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    # The mount folder name depends on the dataset slug, so detect
    # it: any /kaggle/input/* folder holding both csv/ and jpeg/.
    hits = [d for d in sorted(glob.glob('/kaggle/input/*'))
            if os.path.isdir(os.path.join(d, 'csv'))
            and os.path.isdir(os.path.join(d, 'jpeg'))]
    assert hits, ('No CBIS-DDSM dataset (csv/ + jpeg/) found - use '
                  '+ Add Input to attach the awsaf49 CBIS-DDSM dataset.')
    CBIS_ROOT = hits[0]
else:
    from google.colab import files
    print('Upload your kaggle.json:'); files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset
    !unzip -q -o cbis-ddsm-breast-cancer-image-dataset.zip -d data/cbis-ddsm
    CBIS_ROOT = 'data/cbis-ddsm'

print('CBIS-DDSM root:', CBIS_ROOT)
print('csv/ :', sorted(os.listdir(os.path.join(CBIS_ROOT, 'csv'))))

## 3 · Load the dataset (training + test splits)
`CBISDDSMDataset` joins the case-description CSVs to the JPEG mirror and aggregates abnormality rows to one image-level case (`y_true` = malignant if any abnormality is malignant). CBIS-DDSM ships a fixed official split: the **training** split fits the model's linear head, the **test** split is what gets validated.

In [ ]:
from mammoval.data import CBISDDSMDataset
import json

ds_train = CBISDDSMDataset(root=CBIS_ROOT, split='train')
ds_test  = CBISDDSMDataset(root=CBIS_ROOT, split='test')
print('train:', json.dumps(ds_train.summary(), default=str))
print('test :', json.dumps(ds_test.summary(),  default=str))
ds_test.cases.head()

## 4 · Build the model — fine-tuned CNN
A frozen-backbone linear probe is reliable but weak on mammography — ImageNet features transfer poorly to grayscale breast tissue. Here we **fine-tune** an ImageNet-pretrained CNN end-to-end on the CBIS-DDSM **training** split: the convolutional weights are updated, so the network learns mammography-specific features.

Training holds out an internal validation split and keeps the best-epoch weights. This is a genuine (short) training step (~10–15 min on a Kaggle GPU); the pipeline that scores the **test** split afterwards is unchanged. `LinearProbeClassifier` remains available as the faster, no-training baseline.

In [ ]:
from mammoval.models import FineTunedClassifier

model = FineTunedClassifier(backbone='resnet50', image_size=320)
# Fine-tune end-to-end on the TRAINING split. Lower image_size or
# epochs for a faster pass; raise them for a stronger model.
model.fit(ds_train, epochs=6, batch_size=24)
print('fine-tuned:', model.name,
      '| trained on', model.n_train, 'images',
      '| best internal val AUC = %.3f' % model.val_auc)

## 5 · Run inference on the test split → predictions table
`score_dataset` runs the fitted model over every test case and returns the case table with a `y_score` column. Set `LIMIT=None` for the full official test split.

In [ ]:
from mammoval.models import score_dataset

LIMIT = None         # or an integer for a quick pass
preds = score_dataset(model, ds_test, limit=LIMIT)
preds = preds.dropna(subset=['y_score'])
preds[['case_id','y_true','y_score','breast_density_cat',
       'abnormality_type','birads_assessment']].head()

## 6 · Run the clinical validation pipeline
One call produces the full result: discrimination, operating points, calibration, screening behaviour, subgroup analysis, and the paired AI-vs-reader comparison (BI-RADS assessment is used as the radiologist reference).

In [ ]:
from mammoval.pipeline import ClassificationConfig, run_classification_validation

cfg = ClassificationConfig(
    reader_col='birads_assessment',
    subgroup_cols=('breast_density_cat', 'abnormality_type', 'view'),
    target_specificities=(0.90, 0.96),
    ni_margin=0.05,
    is_probability=True,
    dataset_name='CBIS-DDSM test split',
)
results = run_classification_validation(preds, cfg)
d = results['discrimination']
print('AUC = %.3f   95%% CI %.3f-%.3f' % (d['auc'], *d['auc_ci']))

## 7 · Generate the HTML validation report

In [ ]:
from mammoval.report import build_report
from IPython.display import HTML

os.makedirs('outputs', exist_ok=True)
path = build_report(
    results,
    output_path='outputs/cbis_ddsm_validation_report.html',
    title='2D Mammography AI — Clinical Validation (CBIS-DDSM)',
    extra_limitations=[
        'CBIS-DDSM is digitised film, not full-field digital mammography — '
        'a real domain shift from a modern screening device input.',
        'The model is an ImageNet CNN fine-tuned briefly on CBIS-DDSM - '
        'a reasonable baseline, not a state-of-the-art device; the '
        'validation methodology is the deliverable.',
    ])
HTML(open(path, encoding='utf-8').read())

## Notes & next steps
* **Stronger model.** Raise `image_size` / `epochs`, try another backbone, or write one `ImageClassifier` subclass for a different model — the validation is identical.
* **Reader caveat.** BI-RADS *assessment* is an ordinal proxy reader, not an independent radiologist read; category 0 ('incomplete') is not strictly ordered. A true reader study is MRMC (see `docs/methodology.md`).
* **External validation.** Re-run on VinDr-Mammo or RSNA to probe generalisation across vendor / population — the report's CIs cover sampling error only, not distribution shift.